# GPT from Scratch By Andrej Karpathy Youtube Series

In [1]:
# download shakespeare.txt
import requests
url = "https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt"
response = requests.get(url)
with open("shakespeare.txt", "w") as file:
    file.write(response.text)


In [3]:
!ls

chapt1_3.ipynb  gpt.ipynb       part2.ipynb     README.md
chpt1.ipynb     intro.ipynb     part3.ipynb     shakespeare.txt


In [5]:
# read the file
with open("shakespeare.txt", "r") as file:
    text = file.read()

In [6]:
print(text[:1000])  # Print the first 1000 characters to check the content

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



In [7]:
chars = sorted(list(set(text)))
vocab_size = len(chars)
print(vocab_size)
print(''.join(chars))

65

 !$&',-.3:;?ABCDEFGHIJKLMNOPQRSTUVWXYZabcdefghijklmnopqrstuvwxyz


In [8]:
# Mapping characters to integers and vice versa
stoi = {ch: i for i, ch in enumerate(chars)}  # string to integer
itos = {i: ch for i, ch in enumerate(chars)}  # integer to string
encode = lambda s: [stoi[c] for c in s]  # encode a string to a list of integers
decode = lambda l: ''.join([itos[i] for i in l])  # decode a list of integers to a string

print(encode('hello'))
print(decode(encode('hello')))

[46, 43, 50, 50, 53]
hello


In [10]:
# let's encode the entire text
import torch
data = torch.tensor(encode(text), dtype=torch.long)  # encode the text to a tensor of integers
print(data.shape, data.dtype)  # should be (N,) where N is the number of characters in the text
print(data[:1000]) # print the first 1000 integers to check the encoding

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

In [11]:
# Split the data
n = int(0.9 * len(data))  # 90% for training, 10% for validation
train_data = data[:n]
val_data = data[n:]  # the rest for validation

In [13]:
block_size = 8  # context length, how many characters to predict at once
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

In [18]:
decode([18, 47, 56, 57, 58])

'First'

In [20]:
x = train_data[:block_size]
y = train_data[1:block_size+1]  # next character is the target
for t in range(block_size):
    context = x[:t+1]  # context is the first t+1 characters
    target = y[t]  # target is the next character
    print(f'When input is {context}, target is {target}')

When input is tensor([18]), target is 47
When input is tensor([18, 47]), target is 56
When input is tensor([18, 47, 56]), target is 57
When input is tensor([18, 47, 56, 57]), target is 58
When input is tensor([18, 47, 56, 57, 58]), target is 1
When input is tensor([18, 47, 56, 57, 58,  1]), target is 15
When input is tensor([18, 47, 56, 57, 58,  1, 15]), target is 47
When input is tensor([18, 47, 56, 57, 58,  1, 15, 47]), target is 58


In [23]:
torch.manual_seed(1337)  # for reproducibility
batch_size = 4
block_size = 8
def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))  # random starting indices for each batch
    x = torch.stack([data[i:i+block_size] for i in ix])  # shape (batch_size, block_size)
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])  # shape (batch_size, block_size)
    return x, y

xb, yb = get_batch('train')
print('input:', xb)
print('target:', yb)
print('----')
for b in range(batch_size):
    print(f'Batch {b}:')
    for t in range(block_size):
        context = xb[b, :t+1]  # context is the first t+1 characters
        target = yb[b, t]  # target is the next character
        print(f'When input is {context.tolist()}, target is {target.item()}')

input: tensor([[24, 43, 58,  5, 57,  1, 46, 43],
        [44, 53, 56,  1, 58, 46, 39, 58],
        [52, 58,  1, 58, 46, 39, 58,  1],
        [25, 17, 27, 10,  0, 21,  1, 54]])
target: tensor([[43, 58,  5, 57,  1, 46, 43, 39],
        [53, 56,  1, 58, 46, 39, 58,  1],
        [58,  1, 58, 46, 39, 58,  1, 46],
        [17, 27, 10,  0, 21,  1, 54, 39]])
----
Batch 0:
When input is [24], target is 43
When input is [24, 43], target is 58
When input is [24, 43, 58], target is 5
When input is [24, 43, 58, 5], target is 57
When input is [24, 43, 58, 5, 57], target is 1
When input is [24, 43, 58, 5, 57, 1], target is 46
When input is [24, 43, 58, 5, 57, 1, 46], target is 43
When input is [24, 43, 58, 5, 57, 1, 46, 43], target is 39
Batch 1:
When input is [44], target is 53
When input is [44, 53], target is 56
When input is [44, 53, 56], target is 1
When input is [44, 53, 56, 1], target is 58
When input is [44, 53, 56, 1, 58], target is 46
When input is [44, 53, 56, 1, 58, 46], target is 39
When

In [24]:
import torch
import torch.nn as nn
from torch.nn import functional as F
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super(BigramLanguageModel, self).__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  # embedding layer

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # B, T , C
        return logits

m = BigramLanguageModel(vocab_size)
out = m(xb, yb)
print(out.shape)  # should be (batch_size, block_size, vocab_size)

torch.Size([4, 8, 65])


In [26]:
torch.manual_seed(1337)

class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)  # embedding layer

    def forward(self, idx, targets=None):
        logits = self.token_embedding_table(idx) # B, T , C
        B,T,C = logits.shape  # B=batch size, T=block size, C=vocab size
        logits = logits.view(B*T, C)  # reshape to (B*T, C) for cross-entropy loss
        targets = targets.view(B*T)
        loss = F.cross_entropy(logits, targets)
        return logits, loss

m = BigramLanguageModel(vocab_size)
logits, loss = m(xb, yb)
print(logits.shape)
print(loss.item())  # should be a scalar loss value

torch.Size([32, 65])
4.878634929656982
